# Machine Learning Assignment 2
## House Price Prediction with Multiple Linear Regression
Follow Parts A-G in order.

## Part A: Data Exploration and Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df=pd.read_csv("house_price_prediction_dataset.csv")
print("Shape:",df.shape)
display(df.head())
df.info()
display(df.describe(include="all"))

In [ ]:
display(df.isnull().sum().to_frame("Missing Values"))

Drop a row with missing target. Impute numeric predictors with the median and Neighborhood with the mode because only a small number of predictor values are missing.

In [ ]:
print("Duplicates:",df.duplicated().sum())
df=df.drop_duplicates().copy()

In [ ]:
for col in ["Area_m2","House_Age_Years","Distance_to_City_km","House_Price_Million_RWF"]:
    plt.figure(figsize=(6,2.5)); plt.boxplot(df[col].dropna(),vert=False); plt.title(col); plt.show()

After IQR/boxplot inspection, remove House_ID 1006, 1019, 1034, 1048, 1073 and 1092 as exceptionally extreme/inconsistent records. Retain moderate but plausible outliers.

In [ ]:
df=df.dropna(subset=["House_Price_Million_RWF"]).copy()
df=df[~df["House_ID"].isin([1006,1019,1034,1048,1073,1092])].copy()
numeric_features=["Area_m2","Bedrooms","Bathrooms","House_Age_Years","Distance_to_City_km","Parking_Spaces"]
for c in numeric_features: df[c]=df[c].fillna(df[c].median())
df["Neighborhood"]=df["Neighborhood"].fillna(df["Neighborhood"].mode()[0])
df.to_csv("house_price_prediction_cleaned.csv",index=False)
print(df.shape); display(df.isnull().sum())

In [ ]:
plt.hist(df["House_Price_Million_RWF"],bins=15,edgecolor="black");plt.title("Distribution of House Prices");plt.show()
# Interpretation: Most cleaned prices are in the lower-to-middle range, with fewer high-price houses.

In [ ]:
plt.scatter(df["Area_m2"],df["House_Price_Million_RWF"]);plt.xlabel("Area (m²)");plt.ylabel("Price");plt.title("Area vs Price");plt.show()
# Interpretation: Price generally increases as floor area increases.

In [ ]:
names=sorted(df["Neighborhood"].unique()); groups=[df.loc[df["Neighborhood"]==n,"House_Price_Million_RWF"] for n in names]
plt.boxplot(groups,tick_labels=names);plt.xticks(rotation=30);plt.title("Price by Neighborhood");plt.show()
# Interpretation: Price distributions differ across neighborhoods.

In [ ]:
corr=df[numeric_features+["House_Price_Million_RWF"]].corr()
plt.figure(figsize=(8,6));im=plt.imshow(corr,vmin=-1,vmax=1,cmap="coolwarm");plt.colorbar(im);plt.xticks(range(len(corr)),corr.columns,rotation=45,ha="right");plt.yticks(range(len(corr)),corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)): plt.text(j,i,f"{corr.iloc[i,j]:.2f}",ha="center",va="center",fontsize=7)
plt.tight_layout();plt.show()
# Interpretation: Bedrooms and Bathrooms are strongly correlated.

### Cleaning decisions
| Problem | Where | Action | Reason |
|---|---|---|---|
| Missing target | Price | Drop row | Cannot train without target |
| Missing numeric | Predictors | Median impute | Robust and preserves rows |
| Missing category | Neighborhood | Mode impute | Suitable simple categorical treatment |
| Duplicates | Full rows | Remove | Avoid bias |
| Extreme anomalies | IDs 1006,1019,1034,1048,1073,1092 | Remove | Exceptionally extreme/inconsistent |
| Text category | Neighborhood | One-hot; drop first | Regression input and dummy trap |

## Part B: Building the Model

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
features=numeric_features+["Neighborhood"]; target="House_Price_Million_RWF"
X=df[features];y=df[target]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,random_state=42)
preprocessor=ColumnTransformer([("num",SimpleImputer(strategy="median"),numeric_features),("cat",Pipeline([("imputer",SimpleImputer(strategy="most_frequent")),("onehot",OneHotEncoder(drop="first",handle_unknown="ignore"))]),["Neighborhood"])])
model=Pipeline([("preprocessor",preprocessor),("regressor",LinearRegression())])
model.fit(X_train,y_train)

In [ ]:
feature_names=model.named_steps["preprocessor"].get_feature_names_out()
print("Intercept:",model.named_steps["regressor"].intercept_)
display(pd.DataFrame({"Variable":feature_names,"Coefficient":model.named_steps["regressor"].coef_}))

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
numeric_X=df[numeric_features]
display(numeric_X.corr())
display(pd.DataFrame({"Variable":numeric_features,"VIF":[variance_inflation_factor(numeric_X.values,i) for i in range(len(numeric_features))]}))
# Bedrooms and Bathrooms show severe multicollinearity.

## Part C: Evaluation and Interpretation

In [ ]:
from sklearn.metrics import r2_score,mean_absolute_error,mean_squared_error
import math
def adjusted_r2(r2,n,p): return 1-(1-r2)*(n-1)/(n-p-1)
p=len(feature_names); rows=[]
for name,xx,yy in [("Training",X_train,y_train),("Testing",X_test,y_test)]:
    pr=model.predict(xx);r2=r2_score(yy,pr);rows.append([name,r2,adjusted_r2(r2,len(yy),p),mean_absolute_error(yy,pr),math.sqrt(mean_squared_error(yy,pr))])
display(pd.DataFrame(rows,columns=["Dataset","R²","Adjusted R²","MAE","RMSE"]))

In [ ]:
y_pred=model.predict(X_test);plt.scatter(y_test,y_pred);mn=min(y_test.min(),y_pred.min());mx=max(y_test.max(),y_pred.max());plt.plot([mn,mx],[mn,mx],"--");plt.xlabel("Actual");plt.ylabel("Predicted");plt.show()

In [ ]:
residuals=y_test-y_pred;plt.scatter(y_pred,residuals);plt.axhline(0,linestyle="--");plt.xlabel("Predicted");plt.ylabel("Residual");plt.show()
plt.hist(residuals,bins=10,edgecolor="black");plt.title("Residual Histogram");plt.show()

### Interpretation
- Area coefficient 0.560: one extra m² is associated with about 0.560 million RWF higher predicted price, other variables held constant.
- Bathrooms coefficient 19.014: one extra bathroom is associated with about 19.014 million RWF higher predicted price.
- House age coefficient -0.864: each extra year is associated with about 0.864 million RWF lower predicted price.

Linearity is reasonably supported; residuals are centered near zero; constant variance/normality are approximate; severe multicollinearity exists between Bedrooms and Bathrooms. Limitations: small dataset (113 cleaned observations) and omitted price drivers such as exact location, land size and construction quality.

## Part D: Saving the Model

In [ ]:
import joblib
joblib.dump(model,"house_price_model.sav")
loaded_model=joblib.load("house_price_model.sav")
print("Same predictions:",np.allclose(model.predict(X_test),loaded_model.predict(X_test)))

In [ ]:
new_house=pd.DataFrame([{"Area_m2":150.0,"Bedrooms":3,"Bathrooms":2,"House_Age_Years":5.0,"Distance_to_City_km":4.0,"Parking_Spaces":1,"Neighborhood":"Gasabo"}])
print(f"Predicted price: {loaded_model.predict(new_house)[0]:.2f} million RWF")

## Part E: Streamlit App
Use the supplied `app.py`. From Anaconda Prompt in the project folder run: `streamlit run app.py`.

## Part F: GitHub and Deployment
Create public `house-price-predictor`; upload all required files; make at least four meaningful commits; deploy `app.py` on Streamlit Community Cloud; test the public URL.

## Part G: Written Report
Use the supplied PDF. Add screenshots of the local and online app showing a prediction before final submission.